## Neo4j Import of GraphRAG Result Parquet files
This notebook imports the results of the GraphRAG indexing process into the Neo4j Graph database for further processing, analysis or visualization.

### How does it work?
The notebook loads the parquet files from the output folder of your indexing process and loads them into Pandas dataframes. It then uses a batching approach to send a slice of the data into Neo4j to create nodes and relationships and add relevant properties. The id-arrays on most entities are turned into relationships.

All operations use `MERGE`, so they are idempotent, and you can run the script multiple times.

If you need to clean out the database, you can run the following statement
```
MATCH (n)
CALL { WITH n DETACH DELETE n } IN TRANSACTIONS OF 25000 ROWS;
```

In [1]:
GRAPHRAG_FOLDER="."

In [2]:
import pandas as pd
from neo4j import GraphDatabase
import time

In [3]:
from neo4j import GraphDatabase
import time
import os

# Connection details
NEO4J_URI = "neo4j+s://2825da47.databases.neo4j.io"  # Your connection URI
NEO4J_AUTH = (os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))  # Your credentials
NEO4J_DATABASE = "neo4j"  # Usually "neo4j" for default database, not the URI

# Initialize driver
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)

In [ ]:
def batched_import(statement, df, batch_size=1000):
    """
    Import a dataframe into Neo4j using a batched approach.
    Parameters: statement is the Cypher query to execute, df is the dataframe to import, and batch_size is the number of rows to import in each batch.
    """
    total = len(df)
    start_s = time.time()
    for start in range(0,total, batch_size):
        batch = df.iloc[start: min(start+batch_size,total)]
        result = driver.execute_query("UNWIND $rows AS value " + statement, 
                                      rows=batch.to_dict('records'),
                                      database_=NEO4J_DATABASE)
        print(result.summary.counters)
    print(f'{total} rows in { time.time() - start_s} s.')    
    return total

### Indexes and Constraints
Indexes in Neo4j are only used to find the starting points for graph queries, e.g. quickly finding two nodes to connect. Constraints exist to avoid duplicates, we create them mostly on id's of Entity types.

We use some Types as markers with two underscores before and after to distinguish them from the actual entity types.

The default relationship type here is `RELATED` but we could also infer a real relationship-type from the description or the types of the start and end-nodes.

* `__Entity__`
* `__Document__`
* `__Chunk__`
* `__Community__`
* `__Covariate__`

In [ ]:
# MATCH (n)-[r]->(m)
# RETURN n, r, m

In [4]:
# Completely delete all nodes and relationships from the graph
clear_graph_query = "MATCH (n) DETACH DELETE n"
driver.execute_query(clear_graph_query)
print("Graph cleared successfully")

Graph cleared successfully


In [23]:
# Modified constraints - remove the problematic entity_title constraint
statements = """
CREATE CONSTRAINT chunk_id IF NOT EXISTS FOR (c:__Chunk__) REQUIRE c.id IS UNIQUE;
CREATE CONSTRAINT document_id IF NOT EXISTS FOR (d:__Document__) REQUIRE d.id IS UNIQUE;
CREATE CONSTRAINT community_id IF NOT EXISTS FOR (c:__Community__) REQUIRE c.community IS UNIQUE;
CREATE CONSTRAINT entity_id IF NOT EXISTS FOR (e:__Entity__) REQUIRE e.id IS UNIQUE;
CREATE CONSTRAINT covariate_title IF NOT EXISTS FOR (e:__Covariate__) REQUIRE e.title IS UNIQUE;
CREATE CONSTRAINT related_id IF NOT EXISTS FOR ()-[rel:RELATED]->() REQUIRE rel.id IS UNIQUE;
""".split(";")

for statement in statements:
    if len((statement or "").strip()) > 0:
        print(statement)
        driver.execute_query(statement)


CREATE CONSTRAINT chunk_id IF NOT EXISTS FOR (c:__Chunk__) REQUIRE c.id IS UNIQUE


/var/folders/70/pdzh2d9d281f_ylgth2bn0z80000gn/T/ipykernel_9564/2741020919.py:14: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  driver.execute_query(statement)



CREATE CONSTRAINT document_id IF NOT EXISTS FOR (d:__Document__) REQUIRE d.id IS UNIQUE

CREATE CONSTRAINT community_id IF NOT EXISTS FOR (c:__Community__) REQUIRE c.community IS UNIQUE

CREATE CONSTRAINT entity_id IF NOT EXISTS FOR (e:__Entity__) REQUIRE e.id IS UNIQUE

CREATE CONSTRAINT covariate_title IF NOT EXISTS FOR (e:__Covariate__) REQUIRE e.title IS UNIQUE

CREATE CONSTRAINT related_id IF NOT EXISTS FOR ()-[rel:RELATED]->() REQUIRE rel.id IS UNIQUE


In [24]:
# # create constraints, idempotent operation

# statements = """
# create constraint chunk_id if not exists for (c:__Chunk__) require c.id is unique;
# create constraint document_id if not exists for (d:__Document__) require d.id is unique;
# create constraint entity_id if not exists for (c:__Community__) require c.community is unique;
# create constraint entity_id if not exists for (e:__Entity__) require e.id is unique;
# create constraint entity_title if not exists for (e:__Entity__) require e.name is unique;
# create constraint entity_title if not exists for (e:__Covariate__) require e.title is unique;
# create constraint related_id if not exists for ()-[rel:RELATED]->() require rel.id is unique;
# """.split(";")

# for statement in statements:
#     if len((statement or "").strip()) > 0:
#         print(statement)
#         driver.execute_query(statement)

## Import Process
### Importing the Documents
We're loading the parquet file for the documents and create nodes with their ids and add the title property. We don't need to store text_unit_ids as we can create the relationships and the text content is also contained in the chunks.

In [25]:
# !pip install pyarrow


In [ ]:
documents_path = "/Users/juan-garassino/Code/003-personal/004-portfolio/001-achievementsAI/achievements_ai_graphrag/output/documents.parquet"

doc_df = pd.read_parquet(documents_path, columns=["id", "title"])

doc_df#.head(2)

,id,title
0,221759620e72ff46dacef9c01b0137ad75df5e4f5c6bde...,bio_o3.txt
1,8c96e128b48ede9a37d62612a56584c064b5e7e224a259...,juan-garassino-biography.txt
2,37c1cd2140ff2ea9e0735ca04f6489081407da1243f898...,juan-complete-biography.txt
3,554d39ff2c7a6b4574f929f03119ce6c6ec967eb77c40e...,biography-narrative-markdown.txt
4,c07a8adf4f0601360cc79e17612a31376de2b2f253639e...,global-career-journey.txt


In [ ]:
# import documents
statement = """
MERGE (d:__Document__ {id:value.id})
SET d += value {.title}
"""

batched_import(statement, doc_df)

### Loading Text Units
We load the text units, create a node per id and set the text and number of tokens. Then we connect them to the documents that we created before.

In [52]:
path = "/Users/juan-garassino/Code/003-personal/004-portfolio/001-achievementsAI/achievements_ai_graphrag/output/text_units.parquet"

text_df = pd.read_parquet(path, columns=["id","text","n_tokens","document_ids"])

text_df.columns

Index(['id', 'text', 'n_tokens', 'document_ids'], dtype='object')

In [53]:
statement = """
MERGE (c:__Chunk__ {id:value.id})
SET c += value {.text, .n_tokens}
WITH c, value
UNWIND value.document_ids AS document
MATCH (d:__Document__ {id:document})
MERGE (c)-[:PART_OF]->(d)
"""

batched_import(statement, text_df)

{'_contains_updates': True, 'properties_set': 54}
27 rows in 0.787398099899292 s.


27

### Loading Nodes
For the nodes we store id, name, description, embedding (if available), human readable id.

In [55]:
entity_path = "/Users/juan-garassino/Code/003-personal/004-portfolio/001-achievementsAI/achievements_ai_graphrag/output/entities.parquet"

entity_df = pd.read_parquet(entity_path)#, columns=["title","type","description","human_readable_id","id","text_unit_ids"]) # ,"description_embedding" ?!

entity_df#.columns

,id,human_readable_id,title,type,description,text_unit_ids,frequency,degree,x,y
0,06792e45-23e8-4f60-b81e-210d5b19abfb,0,JUAN GARASSINO,PERSON,Juan Garassino is a multifaceted professional ...,[57b0ca0ce66dcdb066b904f5cf9e083bcbfa62ee899f0...,7,54,0.0,0.0
1,8face90f-cbd3-406b-96af-43f51a0725e4,1,BUENOS AIRES,GEO,"Buenos Aires is the capital city of Argentina,...",[57b0ca0ce66dcdb066b904f5cf9e083bcbfa62ee899f0...,14,9,0.0,0.0
2,a38dfa34-56f7-478c-8257-238e96d6fb42,2,UNIVERSIDAD DE BUENOS AIRES,ORGANIZATION,The Universidad de Buenos Aires is a leading a...,[57b0ca0ce66dcdb066b904f5cf9e083bcbfa62ee899f0...,1,2,0.0,0.0
3,4771664c-cf0d-47b3-9457-6872a9a71d65,3,ÉCOLE NATIONALE SUPÉRIEURE D’ARCHITECTURE PARI...,ORGANIZATION,The École Nationale Supérieure d’Architecture ...,[57b0ca0ce66dcdb066b904f5cf9e083bcbfa62ee899f0...,1,1,0.0,0.0
4,97dfb2c6-78e2-4e43-9373-f97a400f91c1,4,BERLIN,GEO,Berlin is a dynamic city in Germany that serve...,[57b0ca0ce66dcdb066b904f5cf9e083bcbfa62ee899f0...,12,13,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
229,c6fc8e2a-0952-425c-bf88-4ffdb611038d,229,ARCHITECTURAL TRAINING,EVENT,The educational background and skills acquired...,[a4a238c5cfaf355993fe91dcabcc2ca765e97f60800c6...,1,1,0.0,0.0
230,bfc49255-3014-48b7-8956-aa4e160615fd,230,AI SOLUTIONS,EVENT,Technological applications developed by the su...,[a4a238c5cfaf355993fe91dcabcc2ca765e97f60800c6...,1,1,0.0,0.0
231,1562f3fe-c200-478f-bf3c-7a9a2a9ed37f,231,UAE,GEO,The UAE is known for its ambitious and resourc...,[35f23ac978c6f3143d0b092e1b19f46138031b8eefc37...,1,1,0.0,0.0
232,76a67cf0-b065-4948-a88d-88f0c89ee732,232,AI CONSULTING,EVENT,AI consulting involves advising clients on the...,[35f23ac978c6f3143d0b092e1b19f46138031b8eefc37...,1,1,0.0,0.0


In [56]:
# Check if description_embedding column exists
if 'description_embedding' in entity_df.columns:
    # Check if all values are null
    all_null = entity_df['description_embedding'].isna().all()
    
    if all_null:
        # Skip setting vector property entirely
        entity_statement = """
        MERGE (e:__Entity__ {id:value.id})
        SET e += value {.human_readable_id, name:replace(value.name,'"','')}
        WITH e, value
        CALL apoc.create.addLabels(e, case when coalesce(value.type,"") = "" then [] else [apoc.text.upperCamelCase(replace(value.type,'"',''))] end) yield node
        UNWIND value.text_unit_ids AS text_unit
        MATCH (c:__Chunk__ {id:text_unit})
        MERGE (c)-[:HAS_ENTITY]->(e)
        """
    else:
        # Use CASE WHEN to conditionally set embedding only when not null
        entity_statement = """
        MERGE (e:__Entity__ {id:value.id})
        SET e += value {.human_readable_id, name:replace(value.name,'"','')}
        WITH e, value
        CALL apoc.util.when(
            value.description_embedding IS NOT NULL,
            'CALL db.create.setNodeVectorProperty($entity, "description_embedding", $embedding)',
            '',
            {entity: e, embedding: value.description_embedding}
        ) YIELD value as _
        CALL apoc.create.addLabels(e, case when coalesce(value.type,"") = "" then [] else [apoc.text.upperCamelCase(replace(value.type,'"',''))] end) yield node
        UNWIND value.text_unit_ids AS text_unit
        MATCH (c:__Chunk__ {id:text_unit})
        MERGE (c)-[:HAS_ENTITY]->(e)
        """
else:
    # Remove vector property procedure entirely
    entity_statement = """
    MERGE (e:__Entity__ {id:value.id})
    SET e += value {.human_readable_id, name:replace(value.name,'"','')}
    WITH e, value
    CALL apoc.create.addLabels(e, case when coalesce(value.type,"") = "" then [] else [apoc.text.upperCamelCase(replace(value.type,'"',''))] end) yield node
    UNWIND value.text_unit_ids AS text_unit
    MATCH (c:__Chunk__ {id:text_unit})
    MERGE (c)-[:HAS_ENTITY]->(e)
    """

# Run the import
batched_import(entity_statement, entity_df)

{'_contains_updates': True, 'labels_added': 234, 'relationships_created': 435, 'nodes_created': 234, 'properties_set': 702}
234 rows in 0.7284419536590576 s.


234

### Import Relationships
For the relationships we find the source and target node by name, using the base `__Entity__` type. After creating the RELATED relationships, we set the description as attribute.

In [ ]:
relationship_path = "/Users/juan-garassino/Code/003-personal/004-portfolio/001-achievementsAI/achievements_ai_graphrag/output/relationships.parquet"

relationship_df = pd.read_parquet(path, columns=["source","target","id","weight","human_readable_id","description","text_unit_ids"]) # ,"rank"

relationship_df.columns

Index(['source', 'target', 'id', 'weight', 'human_readable_id', 'description',
       'text_unit_ids'],
      dtype='object')

In [61]:
rel_statement = """
    MATCH (source:__Entity__ {name:replace(value.source,'"','')})
    MATCH (target:__Entity__ {name:replace(value.target,'"','')})
    // not necessary to merge on id as there is only one relationship per pair
    MERGE (source)-[rel:RELATED {id: value.id}]->(target)
    SET rel += value {.rank, .weight, .human_readable_id, .description, .text_unit_ids}
    RETURN count(*) as createdRels
"""

batched_import(rel_statement, rel_df)

{}
269 rows in 0.5313911437988281 s.


269

### Importing Communities
For communities we import their id, title, level. We connect the `__Community__` nodes to the start and end nodes of the relationships they refer to.

Connecting them to the chunks they orignate from is optional, as the entites are already connected to the chunks.

In [62]:
community_path = "/Users/juan-garassino/Code/003-personal/004-portfolio/001-achievementsAI/achievements_ai_graphrag/output/communities.parquet"

community_df = pd.read_parquet(community_path, 
                     columns=["id","level","title","text_unit_ids","relationship_ids"])

community_df.head(2)

,id,level,title,text_unit_ids,relationship_ids
0,5d89e731-d103-4605-bace-b7adcdad332b,0,Community 0,[2fe7b9eebba568ed937a6cbd487b3ffb1a1d1277ee68d...,"[21c4734f-08c5-4495-8a21-b8f43c1fb153, 2d0595d..."
1,61c6ef99-d3b3-4723-af10-0045bf2df9a0,0,Community 1,[35f23ac978c6f3143d0b092e1b19f46138031b8eefc37...,"[0fdac5b7-6613-4935-9869-6bdc90193c75, 126c1cd..."


In [63]:
statement = """
MERGE (c:__Community__ {community:value.id})
SET c += value {.level, .title}
/*
UNWIND value.text_unit_ids as text_unit_id
MATCH (t:__Chunk__ {id:text_unit_id})
MERGE (c)-[:HAS_CHUNK]->(t)
WITH distinct c, value
*/
WITH *
UNWIND value.relationship_ids as rel_id
MATCH (start:__Entity__)-[:RELATED {id:rel_id}]->(end:__Entity__)
MERGE (start)-[:IN_COMMUNITY]->(c)
MERGE (end)-[:IN_COMMUNITY]->(c)
RETURN count(distinct c) as createdCommunities
"""

batched_import(statement, community_df)

{'_contains_updates': True, 'labels_added': 26, 'nodes_created': 26, 'properties_set': 78}
26 rows in 0.36158013343811035 s.


26

### Importing Community Reports
Fo the community reports we create nodes for each communitiy set the id, community, level, title, summary, rank, and rank_explanation and connect them to the entities they are about. For the findings we create the findings in context of the communities.

In [69]:
community_report_path = "/Users/juan-garassino/Code/003-personal/004-portfolio/001-achievementsAI/achievements_ai_graphrag/output/community_reports.parquet"

community_report_df = pd.read_parquet(community_report_path, columns=["id","community","level","title","summary", "findings","rank","rating_explanation","full_content_json"])

community_report_df.columns

Index(['id', 'community', 'level', 'title', 'summary', 'findings', 'rank',
       'rating_explanation', 'full_content_json'],
      dtype='object')

In [70]:
# import communities
community_statement = """
MERGE (c:__Community__ {community:value.community})
SET c += value {.level, .title, .rank, .rank_explanation, .full_content, .summary}
WITH c, value
UNWIND range(0, size(value.findings)-1) AS finding_idx
WITH c, value, finding_idx, value.findings[finding_idx] as finding
MERGE (c)-[:HAS_FINDING]->(f:Finding {id:finding_idx})
SET f += finding
"""
batched_import(community_statement, community_report_df)

{'_contains_updates': True, 'labels_added': 164, 'relationships_created': 138, 'nodes_created': 164, 'properties_set': 544}
26 rows in 0.381328821182251 s.


26

### Importing Covariates
Covariates are for instance claims on entities, we connect them to the chunks where they originate from.

**By default, covariates are not included in the output, so the file might not exists in your output if you didn't set the configuration to extract claims**

In [18]:
"""
# cov_df = pd.read_parquet(f'{GRAPHRAG_FOLDER}/create_final_covariates.parquet')
# cov_df.head(2)
"""

"\n# cov_df = pd.read_parquet(f'{GRAPHRAG_FOLDER}/create_final_covariates.parquet')\n# cov_df.head(2)\n"

In [19]:
cov_statement = """
MERGE (c:__Covariate__ {id:value.id})
SET c += apoc.map.clean(value, ["text_unit_id", "document_ids", "n_tokens"], [NULL, ""])
WITH c, value
MATCH (ch:__Chunk__ {id: value.text_unit_id})
MERGE (ch)-[:HAS_COVARIATE]->(c)
"""
# batched_import(cov_statement, cov_df)